# v0.36.0 — Vector search

## 1. Intro

Since v0.36.0 a queryset finds the records whose vector is nearest to a query vector, with the full SurrealDB-ORM's
API:

- `VectorField[d]` declares a vector field — a list of `d` numbers, optional and `None` by default — checked on every
  write and every read; `Annotated[list[float] | None, Vector(d)] = None` is its typed form;
- `Index(..., hnsw=True, dimension=d, dist=...)` declares an HNSW vector index, next to the full-text indexes of
  v0.35.0: `index_ddl()` renders it without a connection, and `define_indexes()` applies it only when it is missing or
  changed;
- `similar_to(field, vector, limit)` keeps the `limit` records whose vector is nearest, nearest first, each with its
  distance as `_knn_distance`; `ef=` sizes the HNSW search, and `dist=` is an exact scan that needs no index;
- `similar_to()` is AND-ed with `filter()`, `search()` and a `Subquery` filter, and applies to `count()`, `exists()`,
  the aggregations and the bulk writes.

The vectors are hand-made embeddings of four numbers — one per topic: pets, code, garden, food — so the notebook
downloads no model; an embedding model gives hundreds of numbers, used the same way. The code is the same on
SurrealDB 2.x, 3.x and the embedded engine, and the notebook runs top to bottom on both lines. The outputs below come
from a SurrealDB 3.x server: a 2.x server applies `SEARCH` where §4 shows `FULLTEXT`, prints the same records — its
distances may differ in the last digits —, and prints in §10 the ORM's refusal where 3.x prints scores; §11 says why.

## 2. Connect

`SURREALDB_HOST` and `SURREALDB_PORT` choose the server (default `localhost:8000`), over WebSocket.

In [1]:
import contextlib
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
# WebSocket (.../rpc), as the other notebooks.
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 3. A model with a vector field

`Document`'s `embedding` is a `VectorField[4]`: four numbers, `F32` by default (`VectorField[4, "F64"]` names another
storage type). The model declares two indexes: `nb_doc_embedding`, an HNSW vector index over `embedding` — its
`dimension=` required, its distance `COSINE`, which compares the vectors' directions —, and `nb_doc_title`, the
full-text index of v0.35.0 on `title`, with `highlights=True`: §10 highlights its matches. When the class is defined,
the ORM checks that the vector index and the field agree on the dimension and the storage type.

The cell first removes what an earlier run left — the table, with its indexes, then the analyzer — so the notebook
can be re-executed in place.

In [2]:
from surreal_orm_lite import Analyzer, BaseSurrealModel, Index, SurrealConfigDict, VectorField

PLAIN = Analyzer("nb_plain", ["blank"], ["lowercase"])


class Document(BaseSurrealModel):
    model_config = SurrealConfigDict(
        table_name="nb_doc",
        indexes=[
            Index("nb_doc_embedding", "embedding", hnsw=True, dimension=4, dist="cosine"),
            Index("nb_doc_title", "title", search_analyzer=PLAIN, highlights=True),
        ],
    )

    id: str | None = None
    title: str
    topic: str
    featured: bool = False
    embedding: VectorField[4]


client = await SurrealDBConnectionManager.get_client()
for statement in ("REMOVE TABLE IF EXISTS nb_doc;", "REMOVE ANALYZER IF EXISTS nb_plain;"):
    with contextlib.suppress(Exception):
        await client.query(statement, {})
print(*Document.get_indexes(), sep="\n")

Index('nb_doc_embedding', ('embedding',), hnsw=True, dimension=4, dist='COSINE')
Index('nb_doc_title', ('title',), search_analyzer=Analyzer('nb_plain', ('blank',), ('lowercase',)), highlights=True)


Type checkers do not understand `VectorField[4]`, as in the full ORM. In type-checked code, write the typed form,
which the ORM reads the same way — the same checks and the same column:

```python
from typing import Annotated

from surreal_orm_lite import Vector


class Document(BaseSurrealModel):
    ...
    embedding: Annotated[list[float] | None, Vector(4)] = None
```

## 4. Render and apply the statements

`index_ddl()` renders the statements without a connection: the analyzer, then one `DEFINE INDEX` per index. The ORM
writes every HNSW option — `DIST`, `TYPE`, `EFC` and `M`, SurrealDB's defaults for those not declared — so every
engine stores the same index, and the `HNSW` statement is the same for SurrealDB 2.x and 3.x (the full-text one says
`SEARCH` on 2.x).

`define_table()` (v0.34.0) defines the table first, its `embedding` column an `option<array<float, 4>>`; then
`define_indexes()` applies what is missing or changed and returns the statements applied. A second call returns `[]`:
an unchanged vector index is never rebuilt — a rebuild reads every record of the table.

In [3]:
print(*Document.index_ddl(), sep="\n")
print()
await Document.define_table()
applied = await Document.define_indexes()
print(*applied, sep="\n")
print()
print("a second call:", await Document.define_indexes())

DEFINE ANALYZER OVERWRITE `nb_plain` TOKENIZERS blank FILTERS lowercase;
DEFINE INDEX OVERWRITE `nb_doc_embedding` ON `nb_doc` FIELDS `embedding` HNSW DIMENSION 4 DIST COSINE TYPE F32 EFC 150 M 12;
DEFINE INDEX OVERWRITE `nb_doc_title` ON `nb_doc` FIELDS `title` FULLTEXT ANALYZER `nb_plain` HIGHLIGHTS;

DEFINE ANALYZER OVERWRITE `nb_plain` TOKENIZERS blank FILTERS lowercase;
DEFINE INDEX OVERWRITE `nb_doc_embedding` ON `nb_doc` FIELDS `embedding` HNSW DIMENSION 4 DIST COSINE TYPE F32 EFC 150 M 12;
DEFINE INDEX OVERWRITE `nb_doc_title` ON `nb_doc` FIELDS `title` FULLTEXT ANALYZER `nb_plain` HIGHLIGHTS;

a second call: []


## 5. Eight documents

Eight documents, saved with `bulk_create()`; SurrealDB indexes each vector as it is written. The documents of one topic
point the same way. §9 and §10 search their titles for "python", found in two of the eight — a pet and a programming
book. §10's scores need a term found in fewer than half the documents: SurrealDB 3.x's BM25 gives a term found in half
of them or more no weight (a score of `0.0`).

In [4]:
corpus = [
    ("n1", "Caring for an old cat", "pets", [0.95, 0.0, 0.1, 0.05]),
    ("n2", "Walking the dog in winter", "pets", [0.85, 0.0, 0.3, 0.0]),
    ("n3", "Keeping a python as a pet", "pets", [0.7, 0.45, 0.1, 0.0]),
    ("n4", "Python for beginners", "code", [0.1, 0.9, 0.0, 0.05]),
    ("n5", "Debugging async code", "code", [0.02, 1.0, 0.05, 0.05]),
    ("n6", "Growing tomatoes on a balcony", "garden", [0.0, 0.0, 0.9, 0.35]),
    ("n7", "Pruning fruit trees", "garden", [0.05, 0.0, 1.0, 0.1]),
    ("n8", "Tomato soup in ten minutes", "food", [0.0, 0.0, 0.3, 0.95]),
]
created = await Document.objects().bulk_create(
    [Document(id=key, title=title, topic=topic, embedding=embedding) for key, title, topic, embedding in corpus]
)
print("documents:", len(created))

documents: 8


## 6. Find the nearest

`similar_to("embedding", pets, 3)` keeps the three documents whose embedding is nearest to the query vector `pets`,
nearest first, and sets each one's distance as `_knn_distance` — here the index's `COSINE` distance, computed on the
server. The query vector is bound as a query variable, never written into the statement, and its length is checked
against the field's `VectorField[4]` before any request. The distance is never written back: `model_dump()` and
`save()` leave it out.

In [5]:
pets = [1.0, 0.0, 0.0, 0.0]
code = [0.0, 1.0, 0.0, 0.0]

for document in await Document.objects().similar_to("embedding", pets, 3).exec():
    print(document.title, round(document._knn_distance, 4))
nearest = await Document.objects().similar_to("embedding", pets).first()
print("nearest:", nearest.title, "- never written back:", "_knn_distance" in nearest.model_dump())

Caring for an old cat 0.0069
Walking the dog in winter 0.057
Keeping a python as a pet 0.1648
nearest: Caring for an old cat - never written back: False


## 7. `ef=` and `dist=`

`ef=` sizes the HNSW search (100 by default): a larger value finds the true nearest records more often, at a cost.
The ORM writes it raised to `limit`. `dist=` scans every record instead, exactly, and needs no index: `"cosine"` finds
the same three documents, its distances computed in double precision where the `F32` index computes in single
precision. A query takes `ef=` or `dist=`, not both.

In [6]:
print("ef=40:", [document.title for document in await Document.objects().similar_to("embedding", pets, 3, ef=40).exec()])
exact = await Document.objects().similar_to("embedding", pets, 3, dist="cosine").exec()
print("dist='cosine':", [(document.title, round(document._knn_distance, 4)) for document in exact])

ef=40: ['Caring for an old cat', 'Walking the dog in winter', 'Keeping a python as a pet']
dist='cosine': [('Caring for an old cat', 0.0069), ('Walking the dog in winter', 0.057), ('Keeping a python as a pet', 0.1648)]


## 8. With other conditions

`similar_to()` is AND-ed with `filter()`: the conditions select the records first, then the vector search keeps the
nearest among them — here the code documents nearest to `pets`. It applies to `count()`, and to the bulk writes:
`bulk_update()` writes the `limit` nearest records, whose ids it selects first, and returns their number. A `Subquery`
filter selects the records first too: with `search()` inside it, the documents that match a text, ranked by a vector —
the form a hybrid search builds on.

In [7]:
from surreal_orm_lite import Subquery

nearest_code = Document.objects().filter(topic="code").similar_to("embedding", pets, 2)
print("nearest code documents:", [document.title for document in await nearest_code.exec()])
print("count():", await nearest_code.count())
written = await Document.objects().filter(topic="code").similar_to("embedding", pets, 1).bulk_update(featured=True)
print("bulk_update() wrote:", written)
print("featured:", [document.title for document in await Document.objects().filter(featured=True).exec()])
python_documents = Subquery(Document.objects().search(title="python"))
ranked = Document.objects().filter(id__in=python_documents).similar_to("embedding", code, 2)
print("a Subquery, ranked by code:", [document.title for document in await ranked.exec()])

nearest code documents: ['Python for beginners', 'Debugging async code']
count(): 2
bulk_update() wrote: 1
featured: ['Python for beginners']
a Subquery, ranked by code: ['Python for beginners', 'Keeping a python as a pet']


## 9. With `search()`

`search()` (v0.35.0) is one more condition: `similar_to()` ranks the documents that match the text by the vector. The
nearest documents to `pets` overall do not mention "python" — the first line —, while the second keeps the matches
only, ranked by `pets`: the pet first, then the programming book.

Then what is refused, each before any request: a query vector of another length than the field's `VectorField[4]`
raises `ValueError`, and two combinations raise `SurrealDbError`:

- `as_of()` (v0.26.0): the vector index holds the records' current vectors, so a time-travel read would pick its
  neighbours by their vectors now;
- `live()` and `watch()`: SurrealDB accepts a live query whose condition holds the vector search but never notifies
  it. Read the nearest records with `similar_to()` and watch them by id.

In [8]:
from datetime import timedelta

from surreal_orm_lite import SurrealDbError

print("nearest overall:", [document.title for document in await Document.objects().similar_to("embedding", pets, 2).exec()])
python_by_pets = Document.objects().search(title="python").similar_to("embedding", pets, 2)
print("matching 'python':", [document.title for document in await python_by_pets.exec()])
print()
try:
    Document.objects().similar_to("embedding", [1.0, 0.0, 0.0])
except ValueError as error:
    print(error)
print()
try:
    Document.objects().similar_to("embedding", pets).as_of(timedelta(days=1))
except SurrealDbError as error:
    print(error)
print()
try:
    Document.objects().similar_to("embedding", pets).live()
except SurrealDbError as error:
    print(error)

nearest overall: ['Caring for an old cat', 'Walking the dog in winter']
matching 'python': ['Keeping a python as a pet', 'Python for beginners']

similar_to(): the vector has 3 dimensions; field 'embedding' holds vectors of 4 (VectorField[4]).

similar_to() cannot be combined with as_of(): the vector index holds the records' current vectors, so a time-travel read would pick its neighbours by their vectors now. Run it without as_of().

A live query cannot honour similar_to(): SurrealDB accepts a LIVE SELECT whose condition holds a KNN operator but never notifies it.


## 10. Search annotations next to `similar_to()`

On SurrealDB 3.x, search annotations (`SearchScore`, `SearchHighlight`, `SearchOffsets`) give each of the nearest
matches the score, highlight and offsets that the same `search()` gives it without `similar_to()`, as in the full ORM:
the outputs below print each title, its highlighted text, its BM25 score and its distance. On SurrealDB 2.x and the
embedded engine the ORM refuses them before the statement is sent, since there it selects the matches first (a `LET`)
and the vector search holds no match to score, highlight or locate: there this cell prints the refusal. `count()`,
`exists()` and the aggregations, which return no annotation, still answer on every line — the last line. Keep
`search()` as a condition there (§9), or use SurrealDB 3.x.

In [9]:
from surreal_orm_lite import SearchHighlight, SearchScore

annotated = python_by_pets.annotate(score=SearchScore(0), snippet=SearchHighlight("<b>", "</b>", 0))
try:
    for document in await annotated.exec():
        print(document.title, "|", document.snippet, "|", round(document.score, 4), round(document._knn_distance, 4))
except SurrealDbError as exc:
    print(exc)
print("count():", await annotated.count(), "- exists():", await annotated.exists())

Keeping a python as a pet | Keeping a <b>python</b> as a pet | 0.8295 0.1648
Python for beginners | <b>Python</b> for beginners | 1.0965 0.8897
count(): 2 - exists(): True


## 11. The line

On SurrealDB 3.x the other conditions stay next to the vector search, which SurrealDB applies during the index search.
SurrealDB 2.x and the embedded engine answer too few records, or none, next to many conditions, so there the ORM
selects the records matching them first, in a `LET`, then searches among them — slower on large tables. On every line
the bulk writes act by the ids of the nearest records, selected first. So §8 and §9 print the same records on both
lines, while §10's scores are SurrealDB 3.x's only. HNSW is approximate — with a condition that keeps few records, a
search can return fewer than `limit` records — and `dist=` is the exact scan.

In [10]:
version = await (await SurrealDBConnectionManager.get_client()).version()
print(f"Server: SurrealDB {int(version.rsplit('-', 1)[-1].split('.')[0])}.x")

Server: SurrealDB 3.x


## 12. Cleanup

`remove_index()` removes the vector index by name; `define_indexes()` never removes one. Then the table and the
analyzer go, and the connection is closed.

In [11]:
print(await Document.remove_index("nb_doc_embedding"))
await client.query("REMOVE TABLE IF EXISTS nb_doc;", {})
print(await SurrealDBConnectionManager.remove_analyzer("nb_plain"))
await SurrealDBConnectionManager.close_connection()
print("Done")

REMOVE INDEX IF EXISTS `nb_doc_embedding` ON `nb_doc`;
REMOVE ANALYZER IF EXISTS `nb_plain`;
Done
